Part 1 -- Return-Risk Scoring Pipeline (35 marks)
Flipkart's category-management team wants to flag orders that are statistically likely to be returned before the return happens, so a support agent (and, later, your own Part 3 agent) can proactively check an order's risk. Everything you build here is the fixed input for Part 3 -- you may not swap in a different dataset or a different final model once this Part is graded.


In [ ]:
# Problem statement

#1.Loading Exact Data
# ── Core imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection     import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline            import Pipeline
from sklearn.preprocessing       import StandardScaler, OneHotEncoder
from sklearn.impute              import SimpleImputer
from sklearn.compose             import ColumnTransformer
from sklearn.dummy               import DummyClassifier, DummyRegressor
from sklearn.linear_model        import LogisticRegression
from sklearn.ensemble            import RandomForestClassifier
from sklearn.metrics             import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)



rng = np.random.default_rng(42)
N = 6000

categories = ["Apparel", "Electronics", "Home", "Footwear", "Beauty"]
cat_probs = [0.32, 0.22, 0.18, 0.18, 0.10]
payment_methods = ["COD", "Prepaid_Card", "Prepaid_UPI", "Wallet"]
pay_probs = [0.42, 0.24, 0.24, 0.10]

product_category = rng.choice(categories, size=N, p=cat_probs)
payment_method = rng.choice(payment_methods, size=N, p=pay_probs)

base_price = {
    "Apparel": (400, 2200), "Electronics": (1200, 45000), "Home": (300, 8000),
    "Footwear": (500, 4500), "Beauty": (150, 2500),
}
price_inr = np.round(np.array([rng.uniform(*base_price[c]) for c in product_category]), 0)

discount_pct = np.clip(rng.normal(22, 15, N), 0, 75)
customer_tenure_days = np.clip(rng.exponential(380, N), 1, 2500).round(0)
num_previous_orders = np.clip((customer_tenure_days / 45) + rng.normal(0, 2, N), 0, None).round(0)
base_return_rate = np.clip(rng.beta(1.5, 9, N), 0, 1)
num_previous_returns = np.round(base_return_rate * num_previous_orders).clip(0, num_previous_orders)

delivery_distance_km = np.clip(rng.gamma(3, 90, N), 2, 2200).round(1)
delivery_days = np.clip(rng.normal(4.5, 2.2, N), 1, 21).round(0)
is_weekend_order = rng.integers(0, 2, N)

rating_given = rng.integers(1, 6, N).astype(float)
missing_mask = rng.random(N) < np.where(payment_method == "COD", 0.22, 0.06)
rating_given[missing_mask] = np.nan

fit_risk_cat = np.isin(product_category, ["Apparel", "Footwear"]).astype(float)
prev_return_ratio = np.where(num_previous_orders > 0,
                              num_previous_returns / np.maximum(num_previous_orders, 1), 0)

z = (-2.2 + 1.9 * prev_return_ratio + 0.55 * fit_risk_cat
     + 0.014 * (discount_pct - 20) / 10 + 0.9 * (payment_method == "COD").astype(float)
     + 0.10 * (delivery_days - 4.5) / 2 + 0.30 * (price_inr / base_price["Electronics"][1])
     + 0.05 * is_weekend_order - 0.15 * np.tanh(customer_tenure_days / 500))
prob_return = 1 / (1 + np.exp(-z))
returned = (rng.random(N) < prob_return).astype(int)

df = pd.DataFrame({
    "order_id": np.arange(1, N + 1), "product_category": product_category,
    "price_inr": price_inr, "discount_pct": np.round(discount_pct, 1),
    "payment_method": payment_method, "customer_tenure_days": customer_tenure_days.astype(int),
    "num_previous_orders": num_previous_orders.astype(int),
    "num_previous_returns": num_previous_returns.astype(int),
    "delivery_distance_km": delivery_distance_km, "delivery_days": delivery_days.astype(int),
    "is_weekend_order": is_weekend_order, "rating_given": rating_given, "returned": returned,
})
df.to_csv("orders_dataset.csv", index=False)
print("Rows:", len(df), "| Return rate:", round(df["returned"].mean(), 4))



Rows: 6000 | Return rate: 0.2275


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
#2.Verify the generated data.

#Report: total row count, overall return rate, percentage of missing rating_given values,
#and a table of return rate broken out by product_category
#and separately by payment_method.
#State explicitly whether the missingness pattern in
#rating_given looks like MCAR, MAR, or MNAR,
#and justify your answer from how the column was actually generated
 #(hint: its missingness depends on another observed column).

df = pd.read_csv('orders_dataset.csv')
display(df.head())
df.info
display(df.describe())

# 1. Basic dataset verification
# ---------------------------------------------------------

print("Dataset shape:", df.shape)
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

# 2. Overall return rate
# ---------------------------------------------------------

overall_return_rate = df["returned"].mean() * 100

print("\nOverall return rate:")
print(f"{overall_return_rate:.2f}%")


# 3. Missing rating_given
missing_rating_pct = df["rating_given"].isna().mean() * 100

print("\nMissing rating_given:")
print(f"{missing_rating_pct:.2f}%")


# 4. Missing rating by payment method
#    This helps determine MCAR / MAR / MNAR
missing_by_payment = (
    df.groupby("payment_method")["rating_given"]
    .apply(lambda x: x.isna().mean() * 100)
    .reset_index(name="missing_rating_pct")
)

print("\nMissing rating percentage by payment method:")
print(missing_by_payment)

# 5. Return rate by product category
return_by_category = (
    df.groupby("product_category")["returned"]
    .agg(
        orders="count",
        returned_orders="sum",
        return_rate="mean"
    )
    .reset_index()
)

return_by_category["return_rate"] *= 100

print("\nReturn rate by product category:")
print(return_by_category)

# ---------------------------------------------------------
# 6. Return rate by payment method
# ---------------------------------------------------------

return_by_payment = (
    df.groupby("payment_method")["returned"]
    .agg(
        orders="count",
        returned_orders="sum",
        return_rate="mean"
    )
    .reset_index()
)

return_by_payment["return_rate"] *= 100

print("\nReturn rate by payment method:")
print(return_by_payment)

# ---------------------------------------------------------
# 7. MAR / MCAR / MNAR conclusion
# ---------------------------------------------------------

cod_missing = df.loc[
    df["payment_method"] == "COD",
    "rating_given"
].isna().mean() * 100

non_cod_missing = df.loc[
    df["payment_method"] != "COD",
    "rating_given"
].isna().mean() * 100

gap = cod_missing - non_cod_missing

print("\nMissingness analysis:")
print(f"COD missing rate: {cod_missing:.2f}%")
print(f"Non-COD missing rate: {non_cod_missing:.2f}%")
print(f"Missing-rate gap: {gap:.2f} percentage points")

print("\nConclusion: MAR")

print(
    "Reason: rating_given missingness depends on the observed "
    "payment_method column. The generator assigns approximately "
    "22% missingness probability to COD orders and 6% to non-COD "
    "orders. Therefore the missingness is not MCAR. It is also "
    "not MNAR because the missingness mechanism does not depend "
    "on the unobserved rating_given value itself."
)

In [ ]:
#3.Preprocess without leakage. Using a ColumnTransformer + Pipeline (scikit-learn),
#impute missing numeric values with the median and missing categorical values with the mode,
#one-hot encode product_category and payment_method, and standard-scale the numeric features.
#Fit the pipeline on the training split only, then transform both splits -- never fit on the test split.

# 2. SEPARATE FEATURES AND TARGET
# order_id is only an identifier, so it is not used as a feature.
X = df.drop(columns=["returned", "order_id"])

y = df["returned"]

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())


# ============================================================
# 3. IDENTIFY NUMERIC AND CATEGORICAL FEATURES
# ============================================================

numeric_features = [
    "price_inr",
    "discount_pct",
    "customer_tenure_days",
    "num_previous_orders",
    "num_previous_returns",
    "delivery_distance_km",
    "delivery_days",
    "is_weekend_order",
    "rating_given"
]

categorical_features = [
    "product_category",
    "payment_method"
]


# ============================================================
# 4. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())


# ============================================================
# 5. NUMERIC PIPELINE
# ============================================================

numeric_pipeline = Pipeline(
    steps=[
        # Missing numeric values → median
        ("imputer", SimpleImputer(strategy="median")),

        # Standardize numeric features
        ("scaler", StandardScaler())
    ]
)


# ============================================================
# 6. CATEGORICAL PIPELINE
# ============================================================

categorical_pipeline = Pipeline(
    steps=[
        # Missing categorical values → most frequent value
        ("imputer", SimpleImputer(strategy="most_frequent")),

        # Convert categories into one-hot encoded columns
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)
##
# 7. COLUMN TRANSFORMER
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)


# ============================================================
# 8. FIT PREPROCESSING ONLY ON TRAINING DATA
# ============================================================

preprocessor.fit(X_train)


# ============================================================
# 9. TRANSFORM TRAIN AND TEST DATA
# ============================================================

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)


# ============================================================
# 10. VERIFY RESULTS
# ============================================================

print("\nProcessed training shape:",
      X_train_processed.shape)

print("Processed test shape:",
      X_test_processed.shape)


# ============================================================
# 11. FEATURE NAMES AFTER TRANSFORMATION
# ============================================================

feature_names = preprocessor.get_feature_names_out()

print("\nNumber of processed features:",
      len(feature_names))

print("\nFirst 20 processed feature names:")

for feature in feature_names[:20]:
    print(feature)


# ============================================================
# 12. VERIFY THAT ORIGINAL NULLS EXISTED
# ============================================================

print(
    "\nMissing rating_given before preprocessing:",
    X_train["rating_given"].isna().sum()
)

print(
    "Missing rating_given after preprocessing:",
    pd.isna(X_train_processed).sum()
)

print("\nPreprocessing completed successfully.")

In [ ]:
# Dummy Classifier


# 2. SEPARATE FEATURES AND TARGET
# order_id is only an identifier.
# returned is our target variable.

X = df.drop(columns=["returned", "order_id"])
y = df["returned"]

# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# ============================================================
# 4. CREATE DUMMY CLASSIFIER
# ============================================================

dummy_model = DummyClassifier(
    strategy="most_frequent"
)

# ============================================================
# 5. TRAIN THE BASELINE
# ============================================================

dummy_model.fit(
    X_train,
    y_train
)

# ============================================================
# 6. MAKE PREDICTIONS
# ============================================================

y_pred = dummy_model.predict(X_test)

# ============================================================
# 7. CALCULATE ACCURACY
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)
# ============================================================
# 8. CALCULATE F1 SCORE FOR CLASS 1
# ============================================================

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=1
)
# ============================================================
# 9. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)
# ============================================================
# 10. PRINT RESULTS
# ============================================================

print("\n==============================")
print("DUMMY CLASSIFIER RESULTS")
print("==============================")

print(
    f"\nAccuracy: {accuracy:.4f}"
)

print(
    f"F1-score for returned=1: {f1:.4f}"
)

print("\nConfusion Matrix:")
print(cm)

# ============================================================
# 11. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)